### INTRODUÇÃO

### IMPORTES TOTAIS

In [ ]:
%matplotlib inline 
%load_ext autoreload
%autoreload 2
import os
import time
import numpy as np
import pandas as pd
from IPython.display import display 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
sns.set_theme(style="whitegrid")
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix, 
    classification_report
)
from src.logistic_regression import LogisticRegression

## 1. Tratamento da Baseline e Pré-Processamento

In [ ]:
exemplo_path = r"E:\FCUP\AC1\Projeto\datasets\multiclass_classification\dataset_381_ipums_la_98-small.csv"

# Carregar Dados
df_exemplo = pd.read_csv(exemplo_path)
X_df = df_exemplo.iloc[:, :-1].copy()
y_df = df_exemplo.iloc[:, -1].copy()

# 1. Informações Estruturais
print(f"Dimensões originais - Amostras(Linhas): {X_df.shape[0]}, Features(Colunas): {X_df.shape[1]}")
print(f"Total de valores nulos (NaN): {X_df.isnull().sum().sum()}")

# 2. Análise dos Tipos de Dados
tipos_dados = X_df.dtypes.value_counts()
for tipo, contagem in tipos_dados.items():
    print(f"- {tipo}: {contagem} coluna(s)")

# 3. Verificação do Target (y)
print(f"Tipo de dado do Target (y): {y_df.dtype}")

### Tratamento dos valores Nulos
1. Variáveis Numéricas (Imputação pela Mediana)O que fizemos: Substituímos os valores nulos pelo valor central da coluna (mediana).Porquê: Ao contrário da média, a mediana é robusta a outliers (valores extremos). Num dataset real, onde podem existir ruídos ou valores atípicos, a média pode ser "puxada" para longe do valor real, enquanto a mediana mantém a representatividade estatística da maioria dos dados.

2. Variáveis Categóricas/Texto (Imputação pela Moda)O que fizemos: Substituímos os NaNs pelo valor mais frequente da coluna (moda).Porquê: Para dados do tipo object (strings), não é possível calcular operações matemáticas. A moda é a escolha lógica para manter a consistência categórica, assumindo que a falta de um dado é mais provável pertencer à classe mais comum.

In [ ]:
def imputar_nulos(df):
    """
    Função pura que substitui NaNs pela mediana (numéricos) ou moda (categóricos).
    Retorna o DataFrame limpo e os índices das linhas que foram alteradas.
    """
    df_clean = df.copy()
    indices_nulos = df_clean[df_clean.isnull().any(axis=1)].index
    
    cols_num = df_clean.select_dtypes(include=[np.number]).columns
    cols_cat = df_clean.select_dtypes(exclude=[np.number]).columns
    
    if len(cols_num) > 0:
        df_clean[cols_num] = df_clean[cols_num].fillna(df_clean[cols_num].median())
        
    for col in cols_cat:
        if df_clean[col].isnull().any():
            df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
            
    return df_clean, indices_nulos

### EXEMPLO PRATICO

In [ ]:
# 1. Chamar a função usando o X_df que já está na memória
X_limpo, linhas_alteradas = imputar_nulos(X_df)

# 2. Imprimir os resultados e analisar (EDA)
print(f"Total de NaNs ANTES da limpeza: {X_df.isnull().sum().sum()}")
print(f"Total de NaNs DEPOIS da limpeza: {X_limpo.isnull().sum().sum()}")

if len(linhas_alteradas) > 0:
    print("\n--- ANTES da limpeza (As linhas que tinham NaNs): ---")
    # Mostra o X_df original
    display(X_df.loc[linhas_alteradas].head(5))
    
    print("\n--- DEPOIS da limpeza (As mesmas linhas corrigidas): ---")
    # Mostra o X_limpo
    display(X_limpo.loc[linhas_alteradas].head(5))
else:
    print("\nO dataset não tinha valores nulos.")

### Conversão de Texto para Números (One-Hot Encoding)
O nosso algoritmo matemático não compreende *strings* (variáveis do tipo `object`). Para resolver isto, aplicamos a técnica de **One-Hot Encoding** usando a função `get_dummies` do pandas.

* **O que faz:** Cria uma nova coluna binária (0 ou 1) para cada categoria de texto.
* **Prevenção de Colinearidade:** Utilizamos o parâmetro `drop_first=True` para evitar a "Armadilha das Variáveis Dummy" (*Dummy Variable Trap*), garantindo que as novas colunas não sejam perfeitamente correlacionadas, o que desestabilizaria o modelo matemático.

In [ ]:
def aplicar_encoding(X_df, y_df):
    """
    Função pura: Aplica One-Hot Encoding nas features categóricas (X) e
    converte a variável alvo (y) para numérico. 
    Retorna o DataFrame transformado, as matrizes Numpy e as colunas originais.
    """
    X_clean = X_df.copy()
    y_clean = y_df.copy()
    
    # 1. Identificar as colunas de texto
    cols_cat = X_clean.select_dtypes(exclude=[np.number]).columns
    
    # 2. Aplicar o Encoding
    if len(cols_cat) > 0:
        X_clean = pd.get_dummies(X_clean, columns=cols_cat, drop_first=True)
    
    # 3. Converter tudo para Float (necessário para o Gradient Descent)
    X_clean = X_clean.astype(float)
    X_numpy = X_clean.values
    
    # 4. Tratar a variável alvo (y)
    if y_clean.dtype == 'object':
        y_numpy = pd.factorize(y_clean)[0]
    else:
        y_numpy = y_clean.values
        
    return X_clean, X_numpy, y_numpy, cols_cat

### EXEMPLO PRATICO

In [ ]:
# --- 1. Ver o ANTES ---
cols_cat = X_limpo.select_dtypes(exclude=[np.number]).columns
if len(cols_cat) > 0:
    print("--- ANTES do One-Hot Encoding (Apenas as colunas de texto) ---")
    display(X_limpo[cols_cat].head(5))
else:
    print("--- Não foram encontradas colunas de texto neste dataset. ---")

print(f"\nNúmero de colunas originais: {X_limpo.shape[1]}")

# --- 2. Executar a função ---
# Passamos o X_limpo e o y_df (que carregaste na 1ª célula do Notebook)
X_df_encoded, X_numpy, y_numpy, colunas_cat = aplicar_encoding(X_limpo, y_df)

# --- 3. Ver o DEPOIS ---
print(f"\n--- DEPOIS do One-Hot Encoding ---")
print(f"O número de features (colunas) saltou para {X_df_encoded.shape[1]}!")
print("Como o dataset numérico ficou agora:")
display(X_df_encoded.head(5))

# As tuas variáveis finais prontas para o modelo são 'X_numpy' e 'y_numpy'

### NORMALIZAÇAO

In [ ]:
def normalizar_zscore(X_train, X_test):
    """
    Aplica a escala Z-Score (StandardScaler).
    Retorna os dados escalados e os parâmetros estatísticos.
    """
    # 1. Calcular parâmetros APENAS no treino (evita Data Leakage)
    mu = np.mean(X_train, axis=0)
    sigma = np.std(X_train, axis=0)
    
    # Prevenir divisão por zero caso uma feature seja constante
    sigma[sigma == 0] = 1e-8 
    
    # 2. Aplicar a transformação: z = (x - mu) / sigma
    X_train_s = (X_train - mu) / sigma
    X_test_s = (X_test - mu) / sigma
    
    return X_train_s, X_test_s, mu, sigma


### EXEMPLO PRATICO

In [ ]:
X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
    X_numpy, y_numpy, test_size=0.3, stratify=y_numpy, random_state=42
)

print("--- ANTES da Normalização ---")
print(f"Média da 1ª feature: {np.mean(X_tr_raw[:, 0]):.2f}")
print(f"Desvio Padrão da 1ª feature: {np.std(X_tr_raw[:, 0]):.2f}")

# Chamamos a função (agora os dados são 100% numéricos)
X_train, X_test, mu, sigma = normalizar_zscore(X_tr_raw, X_te_raw)

print("\n--- DEPOIS da Normalização ---")
print(f"Média final: {np.mean(X_train[:, 0]):.10f}")
print(f"Desvio Padrão final: {np.std(X_train[:, 0]):.10f}")

K - FOLD

In [ ]:
def avaliar_kfold(X, y, n_splits=5, lr=0.01, max_iters=1000, random_state=42):
    """
    Função pura: Executa o K-Fold Cross-Validation com normalização local.
    Retorna o DataFrame de métricas, a lista de modelos treinados e os 
    dados da última iteração (para podermos desenhar gráficos depois).
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_results = []
    modelos = []
    
    # Variáveis para guardar os dados do último fold (útil para os gráficos)
    X_te_last, y_te_last, y_pred_last = None, None, None

    for i, (train_idx, test_idx) in enumerate(kf.split(X)):
        # 1. Divisão
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        # 2. Normalização Z-Score (USANDO A NOSSA FUNÇÃO)
        X_tr_s, X_te_s, mu, sigma = normalizar_zscore(X_tr, X_te)

        # 3. Treino
        model = LogisticRegression(lr=lr, max_iters=max_iters)
        model.fit(X_tr_s, y_tr)
        modelos.append(model)

        # 4. Previsão e Métricas
        y_pred = model.predict(X_te_s)
        acc = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred, average='macro', zero_division=0)
        rec = recall_score(y_te, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_te, y_pred, average='macro', zero_division=0)

        fold_results.append({
            'Fold': i + 1,
            'Accuracy': acc, 
            'Precision': prec, 
            'Recall': rec, 
            'F1-Score': f1
        })
        
        # Guardamos sempre os últimos dados processados
        X_te_last, y_te_last, y_pred_last = X_te_s, y_te, y_pred

    # Formatamos a tabela de resultados
    df_folds = pd.DataFrame(fold_results)
    df_folds.set_index('Fold', inplace=True)

    return df_folds, modelos, (X_te_last, y_te_last, y_pred_last)

### EXEMPLO PRATICO

In [ ]:
print("A treinar e avaliar o modelo base com 5-Folds. Por favor aguarda...\n")

# 1. Chamar a função pura (usando o X_numpy e y_numpy que saíram da célula de Encoding)
df_resultados_kf, modelos_treinados, ultimo_fold = avaliar_kfold(X_numpy, y_numpy, n_splits=5)
X_te_last, y_te_last, y_pred_last = ultimo_fold

# 2. Imprimir os resultados (Aqui estão os teus prints do K-Fold!)
print("--- RESULTADOS DETALHADOS (FOLD A FOLD) ---")
display(df_resultados_kf)

print("\n--- PERFORMANCE CONSOLIDADA (Média e Desvio Padrão) ---")
resumo = df_resultados_kf.agg(['mean', 'std']).T
resumo.columns = ['Média (Final)', 'Desvio Padrão']
display(resumo)

# 3. Visualização Gráfica
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico A: Curva de Aprendizagem do último modelo treinado
axes[0].plot(modelos_treinados[-1].errors, color='darkred', linewidth=2)
axes[0].set_title('Curva de Aprendizagem (Último Fold)')
axes[0].set_xlabel('Iterações')
axes[0].set_ylabel('Custo Logístico')
axes[0].grid(True)

# Gráfico B: Matriz de Confusão do último teste
cm = confusion_matrix(y_te_last, y_pred_last)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('Matriz de Confusão (Último Fold)')
axes[1].set_xlabel('Classe Prevista')
axes[1].set_ylabel('Classe Real')

plt.tight_layout()
plt.show()

### Treino, Inferência e Avaliação do Modelo Base
Com os dados limpos e a matriz $X$ perfeitamente padronizada, a matemática da nossa Regressão Logística construída de raiz já pode operar sem sofrer instabilidade (*overflow*).

Vamos instanciar o modelo, cronometrar o treino, calcular as métricas principais e avaliar o desempenho visualmente.

In [ ]:

pastas_datasets = {
    "grupo1_noise": "datasets/noise_outliers/",
    "grupo2_imbalance": "datasets/class_imbalance/",
    "grupo3_multiclass": "datasets/multiclass_classification/"
}

for grupo, path in pastas_datasets.items():
    if not os.path.exists(path):
        print(f"Aviso: Pasta '{path}' não encontrada. A saltar...")
        continue
        
    print(f"\n>>> PROCESSANDO: {grupo.upper()} <<<")
    resultados_grupo = []
    ficheiros = [f for f in os.listdir(path) if f.endswith('.csv')]

    for ficheiro in ficheiros:
        # Usamos end=" " para que o "OK!" ou "FALHOU" apareça na mesma linha do nome do ficheiro
        print(f"  - {ficheiro}...", end=" ") 
        
        meta = {
            'Dataset': ficheiro, 'N_Samples': 0, 'N_Features': 0,
            'Missing_Values': 0, 'Target_Classes': 0, 'Execution_Status': 'Error',
            'Accuracy': 0, 'Precision_Macro': 0, 'Recall_Macro': 0, 'F1_Macro': 0,
            'Training_Time_sec': 0
        }
        
        try:
            t_inicio = time.time()
            
            # 1. Carregar Dados
            df_full = pd.read_csv(os.path.join(path, ficheiro))
            X_orig = df_full.iloc[:, :-1]
            y_orig = df_full.iloc[:, -1]
            
            meta['N_Samples'] = X_orig.shape[0]
            meta['N_Features'] = X_orig.shape[1]
            meta['Missing_Values'] = X_orig.isnull().sum().sum()
            meta['Target_Classes'] = len(np.unique(y_orig))

            # =========================================================
            # A LINHA DE MONTAGEM (Usando as tuas funções puras!)
            # =========================================================
            
            # Passo A: Limpeza
            X_limpo, _ = imputar_nulos(X_orig)
            
            # Passo B: Encoding
            _, X_numpy, y_numpy, _ = aplicar_encoding(X_limpo, y_orig)
            
            # Passo C: K-Fold (A função faz o split, normaliza, treina e avalia)
            df_resultados_kf, _, _ = avaliar_kfold(X_numpy, y_numpy, n_splits=5)
            
            # =========================================================

            # Extrair as médias finais geradas pela nossa tabela de K-Fold
            meta['Execution_Status'] = 'Success'
            meta['Accuracy'] = df_resultados_kf['Accuracy'].mean()
            meta['Precision_Macro'] = df_resultados_kf['Precision'].mean()
            meta['Recall_Macro'] = df_resultados_kf['Recall'].mean()
            meta['F1_Macro'] = df_resultados_kf['F1-Score'].mean()
            
            meta['Training_Time_sec'] = time.time() - t_inicio
            
            print("OK!")

        except Exception as e:
            meta['Execution_Status'] = f"Error: {str(e)}"
            # Imprimimos o erro para saberes exatamente o que falhou se acontecer
            print(f"FALHOU ({str(e)})") 

        resultados_grupo.append(meta)

    # Exportar CSV do grupo
    if resultados_grupo:
        df_result = pd.DataFrame(resultados_grupo)
        df_result.to_csv(f"processado_{grupo}.csv", index=False)
        print(f"=== FICHEIRO processado_{grupo}.csv CRIADO ===")